# PyGeoFetch — 04: Download & Post-Processing

This notebook covers the full download API: band selection, parallel downloads, retry logic, checksum verification, and the post-processing chain (reproject, compress, COG, NDVI, clip).

---
### What you'll learn
- Band selection to reduce download size by 75%
- Parallel downloads with real-time progress
- Resume interrupted downloads
- Checksum verification
- Post-processing: reproject, compress, COG, NDVI
- Inspect downloaded GeoTIFFs
- Download from Planetary Computer (Azure SAS tokens)

In [ ]:
from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import SearchQuery, BoundingBox
from pygeofetch.models.download_task import DownloadOptions, PostProcessAction
from pathlib import Path
import json

client = PyGeoFetch(log_level="INFO")
Path("output/downloads").mkdir(parents=True, exist_ok=True)
print("Ready")

## 1. Find a Clear Scene to Download

In [ ]:
# Search for the clearest available scene
query = SearchQuery(
    bbox=BoundingBox.from_string("-74.1,40.6,-73.7,40.9"),
    start_date="2024-01-01",
    end_date="2024-06-01",
    cloud_cover_max=10,
    max_results=10,
    sort_by="cloud_cover",
    sort_ascending=True,
)

results = client.search(query, providers=["aws_earth"])
print(f"Found {len(results)} scenes")

# Show assets available for the best scene
best = results[0]
print(f"\nBest scene: {best.id}")
print(f"Cloud cover: {best.cloud_cover}%")
print(f"\nAll assets ({len(best.assets)}):")
for key, asset in best.assets.items():
    roles = asset.roles or []
    media = asset.media_type or ""
    print(f"  {key:<15} roles={roles}  type={media[:40]}")

In [ ]:
# Show only downloadable data assets
print(f"Data assets for {best.id}:")
total_size_estimate = 0
for key, asset in best.data_assets.items():
    size_mb = (asset.size_bytes / (1024*1024)) if asset.size_bytes else None
    size_str = f"{size_mb:.0f} MB" if size_mb else "unknown size"
    print(f"  {key:<12} → {size_str}")
    if size_mb:
        total_size_estimate += size_mb

if total_size_estimate:
    print(f"\nEstimated total: {total_size_estimate:.0f} MB")

## 2. Band Selection — Download Only What You Need

In [ ]:
# Band reference for Sentinel-2
sentinel2_bands = {
    "B01": ("Coastal Aerosol", "60m"),
    "B02": ("Blue",            "10m"),
    "B03": ("Green",           "10m"),
    "B04": ("Red",             "10m"),
    "B05": ("Red Edge 1",      "20m"),
    "B06": ("Red Edge 2",      "20m"),
    "B07": ("Red Edge 3",      "20m"),
    "B08": ("NIR",             "10m"),
    "B8A": ("Narrow NIR",      "20m"),
    "B09": ("Water Vapour",    "60m"),
    "B11": ("SWIR 1",          "20m"),
    "B12": ("SWIR 2",          "20m"),
    "SCL": ("Scene Classification", "20m"),
    "visual": ("True Color (TCI)", "10m"),
}

use_cases = {
    "RGB visualization":    ["B02", "B03", "B04"],
    "NDVI (vegetation)": ["B04", "B08"],
    "NDWI (water)":         ["B03", "B08"],
    "False color (urban)":  ["B11", "B08", "B04"],
    "4-band multispectral": ["B02", "B03", "B04", "B08"],
    "SWIR composite":       ["B12", "B11", "B04"],
    "Full scene":           list(sentinel2_bands.keys()),
}

print("Sentinel-2 band combinations for common use cases:")
print(f"{'Use Case':<30} {'Bands':<35} {'~Size'}")
print("-" * 80)
sizes = {"10m": 50, "20m": 15, "60m": 2}
for use_case, bands in use_cases.items():
    size = sum(sizes.get(sentinel2_bands.get(b, ("?", "10m"))[1], 0) for b in bands)
    band_str = ", ".join(bands[:6])
    if len(bands) > 6:
        band_str += f" +{len(bands)-6}"
    print(f"  {use_case:<28} {band_str:<35} ~{size} MB")

In [ ]:
# Download RGB bands only (~150 MB vs ~600 MB for full scene)
options_rgb = DownloadOptions(
    parallel=1,
    retry_attempts=3,
    resume=True,
    verify_checksum=False,
    on_failure="skip",
    bands=["B02", "B03", "B04"],
)

print(f"Downloading RGB bands for: {best.id}")
print(f"  Cloud cover: {best.cloud_cover}%")
print(f"  Bands: B02 (Blue), B03 (Green), B04 (Red)")
print()

dl_results = client.download(
    [best],
    destination=Path("output/downloads/rgb"),
    options=options_rgb,
)

for dr in dl_results:
    if dr.success:
        size_mb = dr.bytes_downloaded / (1024*1024) if dr.bytes_downloaded else 0
        print(f"✓ Downloaded: {dr.data_id}")
        print(f"  Size: {size_mb:.1f} MB | Time: {dr.duration_seconds:.1f}s | Speed: {dr.speed_mbps:.2f} Mbps")
        for p in dr.output_paths:
            print(f"  File: {p.name}")
    else:
        print(f"✗ Failed: {dr.error}")

## 3. Download Options — Full Reference

In [ ]:
# All DownloadOptions fields
full_options = DownloadOptions(
    parallel=4,              # Concurrent download workers
    retry_attempts=5,        # Max retries per file
    retry_delay_seconds=2.0, # Base delay (doubles each retry)
    verify_checksum=True,    # SHA256 verification after download
    resume=True,             # Resume interrupted downloads
    bandwidth_limit_mbps=10, # Throttle to 10 MB/s (0 = unlimited)
    overwrite=False,         # Skip files that already exist
    on_failure="skip",       # "skip", "abort", or "retry"
    bands=["B02", "B03", "B04"],  # Specific bands
    priority=9,              # 1-10, higher = download sooner
    chunk_size_mb=8.0,       # Download chunk size
    timeout_seconds=300,     # Request timeout
)

# Show all settings
print("DownloadOptions configuration:")
for field, value in full_options.model_dump().items():
    if not field.startswith('_'):
        print(f"  {field:<30} {value}")

## 4. Parallel Download — Multiple Scenes

In [ ]:
# Download 3 scenes in parallel
import time

options_parallel = DownloadOptions(
    parallel=2,          # 2 concurrent downloads
    retry_attempts=3,
    resume=True,
    on_failure="skip",
    bands=["B04", "B08"],  # NDVI bands only
)

scenes_to_download = results[:3]  # Download 3 scenes
print(f"Downloading {len(scenes_to_download)} scenes (NDVI bands) in parallel...")
print("Scenes:")
for s in scenes_to_download:
    print(f"  {s.id}  cloud={s.cloud_cover:.0f}%")
print()

t0 = time.time()
dl_results = client.download(
    scenes_to_download,
    destination=Path("output/downloads/ndvi"),
    options=options_parallel,
)
elapsed = time.time() - t0

# Summary
succeeded = [r for r in dl_results if r.success]
failed = [r for r in dl_results if not r.success]
total_mb = sum(r.bytes_downloaded for r in succeeded if r.bytes_downloaded) / (1024*1024)

print(f"\nDownload complete in {elapsed:.1f}s:")
print(f"  Succeeded: {len(succeeded)}")
print(f"  Failed:    {len(failed)}")
print(f"  Total:     {total_mb:.1f} MB")
for dr in succeeded:
    mb = dr.bytes_downloaded / (1024*1024) if dr.bytes_downloaded else 0
    print(f"  ✓ {dr.data_id[:40]} — {mb:.1f} MB in {dr.duration_seconds:.1f}s")

## 5. Post-Processing Chain

In [ ]:
# Post-processing chain: unzip → reproject → compress → COG
options_pp = DownloadOptions(
    parallel=1,
    resume=True,
    bands=["B02", "B03", "B04"],
    post_process=[
        PostProcessAction(action="reproject", params={"value": "EPSG:4326"}),
        PostProcessAction(action="compress",  params={"value": "lzw"}),
        PostProcessAction(action="cog"),      # Cloud Optimized GeoTIFF
    ]
)

print("Post-processing chain:")
print("  1. Download RGB bands (B02, B03, B04)")
print("  2. Reproject → EPSG:4326 (WGS84)")
print("  3. Compress → LZW")
print("  4. Convert → Cloud Optimized GeoTIFF (COG)")
print()
print("Requires: pip install rasterio")
print()

# Only run if rasterio is installed
try:
    import rasterio
    print(f"rasterio {rasterio.__version__} found — running post-processing")
    dl_pp = client.download([best], destination=Path("output/downloads/processed"), options=options_pp)
    for dr in dl_pp:
        if dr.success:
            print(f"✓ {dr.data_id} — post-processing complete")
            print(f"  post_process_completed: {dr.post_process_completed}")
        else:
            print(f"✗ {dr.error}")
except ImportError:
    print("rasterio not installed — skipping post-processing demo")
    print("Install with: pip install rasterio")

## 6. Inspect Downloaded Files

In [ ]:
# List all downloaded files
dl_root = Path("output/downloads")
all_files = sorted(dl_root.rglob("*.tif"))
total_size = sum(f.stat().st_size for f in all_files)

print(f"Downloaded {len(all_files)} file(s)  ({total_size/(1024*1024):.1f} MB total):")
for f in all_files:
    size_mb = f.stat().st_size / (1024*1024)
    relative = f.relative_to(dl_root)
    print(f"  {str(relative):<60} {size_mb:>8.1f} MB")

In [ ]:
# Read file metadata with rasterio
try:
    import rasterio

    tif_files = sorted(Path("output/downloads").rglob("*.tif"))
    if tif_files:
        f = tif_files[0]
        print(f"Inspecting: {f.name}")
        with rasterio.open(f) as src:
            print(f"  Bands:       {src.count}")
            print(f"  Width×Height:{src.width} × {src.height} pixels")
            print(f"  CRS:         {src.crs}")
            print(f"  Bounds:      {src.bounds}")
            print(f"  Resolution:  {src.res}")
            print(f"  Dtype:       {src.dtypes[0]}")
            print(f"  Nodata:      {src.nodata}")
            print(f"  Driver:      {src.driver}")
            print(f"  Compression: {src.profile.get('compress', 'none')}")
            print(f"  Tiled:       {src.profile.get('tiled', False)}")
    else:
        print("No .tif files found yet — run a download first")
except ImportError:
    print("rasterio not installed — metadata inspection unavailable")

In [ ]:
# Visualize downloaded bands as RGB composite
try:
    import rasterio
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.colors import Normalize

    # Find the B02, B03, B04 files
    dl_dir = Path("output/downloads")
    b04 = next((f for f in dl_dir.rglob("*B04*.tif")), None)
    b03 = next((f for f in dl_dir.rglob("*B03*.tif")), None)
    b02 = next((f for f in dl_dir.rglob("*B02*.tif")), None)

    if b04 and b03 and b02:
        def read_band(path):
            with rasterio.open(path) as src:
                data = src.read(1).astype(np.float32)
                data[data == src.nodata] = np.nan if src.nodata else data
                return data

        red   = read_band(b04)
        green = read_band(b03)
        blue  = read_band(b02)

        # Normalize for display (2nd/98th percentile stretch)
        def stretch(band):
            p2, p98 = np.nanpercentile(band, [2, 98])
            return np.clip((band - p2) / (p98 - p2 + 1e-10), 0, 1)

        rgb = np.stack([stretch(red), stretch(green), stretch(blue)], axis=-1)

        fig, ax = plt.subplots(1, 1, figsize=(10, 10))
        ax.imshow(rgb)
        ax.set_title(f"RGB Composite — {best.id[:40]}\nCloud cover: {best.cloud_cover}%", fontsize=12)
        ax.axis('off')
        plt.tight_layout()
        plt.savefig("output/rgb_composite.png", dpi=150, bbox_inches='tight')
        plt.show()
        print("✓ RGB composite saved → output/rgb_composite.png")
    else:
        print("Download the RGB bands first (B02, B03, B04)")
except ImportError:
    print("Install rasterio + matplotlib for visualization")

In [ ]:
# CLI equivalents
print("CLI equivalents:")
print()
print("# Basic download")
print('PyGeoFetch download run \\')
print('  --from-search output/nyc_results.geojson \\')
print('  --output output/downloads/ \\')
print('  --parallel 2 --max-items 3 \\')
print('  --bands "B02,B03,B04"')
print()
print("# With post-processing")
print('PyGeoFetch download run \\')
print('  --from-search output/nyc_results.geojson \\')
print('  --output output/processed/ \\')
print('  --bands "B04,B08" \\')
print('  --post-process "reproject:EPSG:4326,compress:lzw,cog"')
print()
print("# Full options")
print('PyGeoFetch download run \\')
print('  --from-search output/nyc_results.geojson \\')
print('  --output output/downloads/ \\')
print('  --parallel 4 --retry 5 \\')
print('  --verify-checksum --resume \\')
print('  --bandwidth-limit 10MB \\')
print('  --on-failure skip \\')
print('  --notify webhook:https://hooks.slack.com/YOUR/WEBHOOK')

---
## Summary
- ✅ Band selection: reduced download size by 75% (150MB vs 600MB)
- ✅ Parallel downloads: 3 scenes with 2 workers
- ✅ Resume, retry, and checksum verification
- ✅ Post-processing: reproject → compress → COG
- ✅ Inspected and visualized downloaded GeoTIFFs

**Next:** `05_pipelines_and_scheduling.ipynb` — YAML pipelines, scheduling, and multi-step workflows.